In [3]:
import os
import re
import time
import random
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# 1. Xác định đường dẫn Data/raw/crawled_hanoi.csv
raw_dir = "../Data/raw" if os.path.exists("../Data/raw") else ("../data/raw" if os.path.exists("../data/raw") else "Data/raw")
os.makedirs(raw_dir, exist_ok=True)
output_csv = os.path.join(raw_dir, "crawled_hanoi.csv")

# 2. Nạp lại 449 tin đã crawl trước đó để cào nối tiếp (không mất dữ liệu cũ)
records = []
seen_detail_urls = set()

if os.path.exists(output_csv):
    df_old = pd.read_csv(output_csv)
    if "Link trang con" in df_old.columns:
        df_old = df_old.dropna(subset=["Link trang con"]).drop_duplicates(subset=["Link trang con"])
        records = df_old.to_dict("records")
        seen_detail_urls = set(df_old["Link trang con"].astype(str))
        print(f"♻️ Đã nạp lại {len(records)} tin hợp lệ có sẵn từ crawled_hanoi.csv.")

# 3. Cấu hình Chrome Selenium (mở giao diện thật để bạn thao tác xác minh khi cần)
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
TARGET_TOTAL = 1150  # Mục tiêu > 1.000 tin duy nhất theo đề bài

HANOI_DISTRICTS = [
    "Ba Đình", "Hoàn Kiếm", "Tây Hồ", "Long Biên", "Cầu Giấy", "Đống Đa",
    "Hai Bà Trưng", "Hoàng Mai", "Thanh Xuân", "Nam Từ Liêm", "Bắc Từ Liêm",
    "Hà Đông", "Sơn Tây", "Ba Vì", "Chương Mỹ", "Đan Phượng", "Đông Anh",
    "Gia Lâm", "Hoài Đức", "Mê Linh", "Mỹ Đức", "Phú Xuyên", "Phúc Thọ",
    "Quốc Oai", "Sóc Sơn", "Thạch Thất", "Thanh Oai", "Thanh Trì", "Thường Tín", "Ứng Hòa"
]

def detect_district(text):
    if not text:
        return None
    low = text.lower()
    for d in HANOI_DISTRICTS:
        if d.lower() in low:
            return d
    return None

def detect_house_type(text, default_type="Nhà riêng"):
    if not text:
        return default_type
    low = text.lower()
    if "chung cư" in low or "căn hộ" in low:
        return "Chung cư"
    if "biệt thự" in low or "liền kề" in low:
        return "Biệt thự liền kề"
    if "mặt phố" in low or "mặt tiền" in low:
        return "Nhà mặt phố"
    if "đất" in low:
        return "Đất nền"
    return default_type

def extract_bathrooms(text):
    if not text:
        return None
    m = re.search(r"(\d+)\s*(?:wc|vs|vệ sinh|toilet)", text, re.IGNORECASE)
    return m.group(1) if m else None

def extract_project(text):
    if not text:
        return None
    m = re.search(
        r"(Vinhomes\s+[\w\s]+?|Times City|Royal City|Ciputra|Starlake|Ecopark|KĐT\s+[^,\-\.]+)",
        text, re.IGNORECASE
    )
    return m.group(0).strip() if m else None

def save_checkpoint():
    if records:
        pd.DataFrame(records).drop_duplicates(subset=["Link trang con"]).to_csv(
            output_csv, index=False, encoding="utf-8-sig"
        )

def wait_for_human_if_blocked(soup, selector_check, current_url):
    """Kiểm tra nếu trang hiện CAPTCHA hoặc chưa tải danh sách tin thì dừng chờ người dùng xác minh."""
    items = soup.select(selector_check)
    page_low = driver.page_source.lower()
    is_captcha = any(k in page_low for k in ["captcha", "verify you are human", "xác minh", "just a moment", "cf-challenge"])

    if not items or is_captcha:
        print(f"\n🛑 [CẦN XÁC MINH CON NGƯỜI] Trang: {current_url}")
        print("👉 Bước 1: Mở cửa sổ Chrome đang bật, thao tác xác minh bằng tay cho đến khi thấy danh sách nhà hiện ra.")
        user_cmd = input("👉 Bước 2: Bấm vào ô nhập ở TRÊN CÙNG VS Code rồi nhấn ENTER để cào tiếp (hoặc gõ 'skip' để chuyển mục khác): ")
        if user_cmd.strip().lower() == "skip":
            return None
        time.sleep(1.5)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select(selector_check)
    return items

try:
    # ==========================================================================
    # NGUỒN 1: BATDONGSAN.COM.VN (Có dừng chờ bạn xác minh Cloudflare bằng tay)
    # ==========================================================================
    bds_categories = [
        ("Nhà đất bán Hà Nội", "https://batdongsan.com.vn/ban-nha-dat-ha-noi", "Nhà riêng"),
        ("Nhà riêng Hà Nội", "https://batdongsan.com.vn/ban-nha-rieng-ha-noi", "Nhà riêng"),
        ("Chung cư Hà Nội", "https://batdongsan.com.vn/ban-can-ho-chung-cu-ha-noi", "Chung cư"),
    ]

    for cat_name, base_url, default_type in bds_categories:
        if len(records) >= TARGET_TOTAL:
            break
        print(f"\n🚀 [Batdongsan.com.vn - {cat_name}] Bắt đầu cào... (Hiện có: {len(records)} tin)")

        for page in range(1, 21):
            if len(records) >= TARGET_TOTAL:
                break

            list_url = base_url if page == 1 else f"{base_url}/p{page}"
            driver.get(list_url)
            time.sleep(random.uniform(3.0, 4.5))

            soup = BeautifulSoup(driver.page_source, "html.parser")
            cards = wait_for_human_if_blocked(soup, "div.js__card", list_url)

            if cards is None:
                print(f"   ⏭️ Đã bỏ qua chuyên mục {cat_name}.")
                break
            if not cards:
                print(f"   ⚠️ Trang {page} không còn tin, chuyển chuyên mục tiếp theo...")
                break

            driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.6);")
            time.sleep(0.8)
            soup = BeautifulSoup(driver.page_source, "html.parser")
            cards = soup.select("div.js__card")

            added = 0
            for card in cards:
                a_tag = card.select_one("a.js__product-link-for-product-id") or card.find("a", href=True)
                if not a_tag or not a_tag.has_attr("href"):
                    continue
                href = a_tag["href"].strip()
                detail_url = f"https://batdongsan.com.vn{href}" if href.startswith("/") else href
                if detail_url in seen_detail_urls:
                    continue
                seen_detail_urls.add(detail_url)

                id_match = re.search(r"-pr(\d+)", detail_url)
                title_tag = card.select_one(".pr-title, .js__card-title, h3")
                title = title_tag.get_text(" ", strip=True) if title_tag else a_tag.get("title", "").strip()
                price_tag = card.select_one(".re__card-config-price")
                area_tag = card.select_one(".re__card-config-area")
                bed_tag = card.select_one(".re__card-config-bedroom")
                bath_tag = card.select_one(".re__card-config-toilet")
                loc_tag = card.select_one(".re__card-location")
                desc_tag = card.select_one(".re__card-description")
                date_tag = card.select_one(".re__card-published-info-published-at")

                location_text = loc_tag.get_text(" ", strip=True) if loc_tag else ""
                description = desc_tag.get_text(" ", strip=True) if desc_tag else None
                full_text = f"{title} {description or ''} {location_text}"

                records.append({
                    "Nguồn dữ liệu": "Crawl_Batdongsan.com.vn",
                    "Mã bản ghi theo nguồn": id_match.group(1) if id_match else card.get("prid"),
                    "Link trang tổng": list_url,
                    "Link trang con": detail_url,
                    "Thời điểm thu thập": collected_at,
                    "Tiêu đề": title,
                    "Ngày đăng": (date_tag.get("aria-label") or date_tag.get_text(" ", strip=True)) if date_tag else None,
                    "Mô tả": description,
                    "Thành phố": "Hà Nội",
                    "Khu vực/Quận huyện theo nguồn": detect_district(full_text),
                    "Phường xã": location_text.split(",")[0].strip() if "," in location_text else None,
                    "Tên dự án nếu có": extract_project(full_text),
                    "Loại nhà": detect_house_type(title, default_type),
                    "Diện tích": area_tag.get_text(" ", strip=True) if area_tag else None,
                    "Số tầng": None,
                    "Số phòng ngủ": bed_tag.get_text(" ", strip=True) if bed_tag else None,
                    "Số phòng vệ sinh": bath_tag.get_text(" ", strip=True) if bath_tag else extract_bathrooms(full_text),
                    "Giá gốc dạng văn bản": price_tag.get_text(" ", strip=True) if price_tag else None,
                })
                added += 1

            save_checkpoint()
            print(f"   ✓ Batdongsan ({cat_name} - trang {page}): +{added} tin | Tổng: {len(records)}/{TARGET_TOTAL}")

    # ==========================================================================
    # NGUỒN 2: ALONHADAT.COM.VN (Có dừng chờ bạn xác minh CAPTCHA bằng tay)
    # ==========================================================================
    alonhadat_categories = [
        ("Nhà mặt tiền", "https://alonhadat.com.vn/nha-dat/can-ban/nha-mat-tien/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/nha-mat-tien/1/ha-noi/trang--{page}.html", "Nhà mặt phố"),
        ("Nhà trong hẻm", "https://alonhadat.com.vn/nha-dat/can-ban/nha-trong-hem/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/nha-trong-hem/1/ha-noi/trang--{page}.html", "Nhà riêng"),
        ("Căn hộ chung cư", "https://alonhadat.com.vn/nha-dat/can-ban/can-ho-chung-cu/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/can-ho-chung-cu/1/ha-noi/trang--{page}.html", "Chung cư"),
        ("Biệt thự liền kề", "https://alonhadat.com.vn/nha-dat/can-ban/biet-thu-nha-lien-ke/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/biet-thu-nha-lien-ke/1/ha-noi/trang--{page}.html", "Biệt thự liền kề"),
        ("Đất thổ cư", "https://alonhadat.com.vn/nha-dat/can-ban/dat-tho-cu-dat-o/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/dat-tho-cu-dat-o/1/ha-noi/trang--{page}.html", "Đất nền"),
        ("Nhà đất tổng hợp", "https://alonhadat.com.vn/nha-dat/can-ban/nha-dat/1/ha-noi.html", "https://alonhadat.com.vn/nha-dat/can-ban/nha-dat/1/ha-noi/trang--{page}.html", "Nhà riêng"),
    ]

    for cat_name, url_p1, url_tpl, default_type in alonhadat_categories:
        if len(records) >= TARGET_TOTAL:
            break
        print(f"\n🚀 [Alonhadat.com.vn - {cat_name}] Bắt đầu cào... (Hiện có: {len(records)} tin)")

        for page in range(1, 26):
            if len(records) >= TARGET_TOTAL:
                break

            list_url = url_p1 if page == 1 else url_tpl.format(page=page)
            driver.get(list_url)
            time.sleep(random.uniform(2.0, 3.2))

            soup = BeautifulSoup(driver.page_source, "html.parser")
            items = wait_for_human_if_blocked(soup, "div.content-item, article.property-item", list_url)

            if items is None:
                print(f"   ⏭️ Đã bỏ qua chuyên mục {cat_name}.")
                break
            if not items:
                print(f"   ⚠️ Hết tin ở trang {page}, chuyển chuyên mục tiếp theo...")
                break

            added = 0
            for item in items:
                a_tag = item.select_one(".ct_title a, h3 a, a[href$='.html']")
                if not a_tag or not a_tag.has_attr("href"):
                    continue
                href = a_tag["href"].strip()
                detail_url = f"https://alonhadat.com.vn{href}" if href.startswith("/") else href
                if detail_url in seen_detail_urls:
                    continue
                seen_detail_urls.add(detail_url)

                source_record_id = href.split("-")[-1].replace(".html", "") if "-" in href else None
                title = a_tag.get_text(" ", strip=True)
                brief_tag = item.select_one(".ct_brief, .property-description")
                description = brief_tag.get_text(" ", strip=True) if brief_tag else None
                date_tag = item.select_one(".ct_date, .property-date")
                posted_date = date_tag.get_text(" ", strip=True) if date_tag else None
                dis_tag = item.select_one(".ct_dis, .property-address")
                location_text = dis_tag.get_text(" ", strip=True) if dis_tag else ""
                dt_tag = item.select_one(".ct_dt, .property-area")
                area_raw = dt_tag.get_text(" ", strip=True) if dt_tag else None
                floor_tag = item.select_one("span.floors, .floors")
                floors_raw = floor_tag.get_text(" ", strip=True) if floor_tag else None
                bed_tag = item.select_one("span.bedroom, .bedroom")
                bedrooms_raw = bed_tag.get_text(" ", strip=True) if bed_tag else None
                price_tag = item.select_one(".ct_price, .property-price")
                raw_price_text = price_tag.get_text(" ", strip=True) if price_tag else None

                full_text = f"{title} {description or ''} {location_text}"
                parts = [p.strip() for p in location_text.split(",") if p.strip()]
                ward = parts[1] if len(parts) >= 3 else (parts[0] if len(parts) == 2 else None)

                records.append({
                    "Nguồn dữ liệu": "Crawl_Alonhadat.com.vn",
                    "Mã bản ghi theo nguồn": source_record_id,
                    "Link trang tổng": list_url,
                    "Link trang con": detail_url,
                    "Thời điểm thu thập": collected_at,
                    "Tiêu đề": title,
                    "Ngày đăng": posted_date,
                    "Mô tả": description,
                    "Thành phố": "Hà Nội",
                    "Khu vực/Quận huyện theo nguồn": detect_district(full_text),
                    "Phường xã": ward,
                    "Tên dự án nếu có": extract_project(full_text),
                    "Loại nhà": detect_house_type(title, default_type),
                    "Diện tích": area_raw,
                    "Số tầng": floors_raw,
                    "Số phòng ngủ": bedrooms_raw,
                    "Số phòng vệ sinh": extract_bathrooms(full_text),
                    "Giá gốc dạng văn bản": raw_price_text,
                })
                added += 1

            save_checkpoint()
            print(f"   ✓ Alonhadat ({cat_name} - trang {page}): +{added} tin | Tổng: {len(records)}/{TARGET_TOTAL}")

finally:
    driver.quit()

df_crawled = pd.DataFrame(records).drop_duplicates(subset=["Link trang con"])
df_crawled.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"\n🎉 HOÀN TẤT BƯỚC 1! Đã lưu {len(df_crawled):,} tin duy nhất tại: {output_csv}")
print("\nThống kê số lượng tin theo từng trang crawl:")
print(df_crawled["Nguồn dữ liệu"].value_counts())
print("\nDanh sách cột đã lưu:")
print(df_crawled.columns.tolist())
display(df_crawled[["Nguồn dữ liệu", "Link trang tổng", "Link trang con", "Tiêu đề", "Diện tích", "Giá gốc dạng văn bản"]].head(3))

♻️ Đã nạp lại 449 tin hợp lệ có sẵn từ crawled_hanoi.csv.

🚀 [Batdongsan.com.vn - Nhà đất bán Hà Nội] Bắt đầu cào... (Hiện có: 449 tin)

🛑 [CẦN XÁC MINH CON NGƯỜI] Trang: https://batdongsan.com.vn/ban-nha-dat-ha-noi
👉 Bước 1: Mở cửa sổ Chrome đang bật, thao tác xác minh bằng tay cho đến khi thấy danh sách nhà hiện ra.
   ✓ Batdongsan (Nhà đất bán Hà Nội - trang 1): +0 tin | Tổng: 449/1150

🛑 [CẦN XÁC MINH CON NGƯỜI] Trang: https://batdongsan.com.vn/ban-nha-dat-ha-noi/p2
👉 Bước 1: Mở cửa sổ Chrome đang bật, thao tác xác minh bằng tay cho đến khi thấy danh sách nhà hiện ra.
   ⚠️ Trang 2 không còn tin, chuyển chuyên mục tiếp theo...

🚀 [Batdongsan.com.vn - Nhà riêng Hà Nội] Bắt đầu cào... (Hiện có: 449 tin)

🛑 [CẦN XÁC MINH CON NGƯỜI] Trang: https://batdongsan.com.vn/ban-nha-rieng-ha-noi
👉 Bước 1: Mở cửa sổ Chrome đang bật, thao tác xác minh bằng tay cho đến khi thấy danh sách nhà hiện ra.
   ⚠️ Trang 1 không còn tin, chuyển chuyên mục tiếp theo...

🚀 [Batdongsan.com.vn - Chung cư Hà Nội

,Nguồn dữ liệu,Link trang tổng,Link trang con,Tiêu đề,Diện tích,Giá gốc dạng văn bản
0,Crawl_Batdongsan.com.vn,https://batdongsan.com.vn/ban-nha-dat-ha-noi,https://batdongsan.com.vn/ban-nha-rieng-phuong...,"Tôi là chủ, cập nhật giá mới: Cần bán đất tặng...","108,9 m²",21 tỷ
1,Crawl_Batdongsan.com.vn,https://batdongsan.com.vn/ban-nha-dat-ha-noi,https://batdongsan.com.vn/ban-nha-biet-thu-lie...,"Bán gấp biệt thự Ciputra 230m2 mặt đường 30m, ...",230 m²,"66,7 tỷ"
2,Crawl_Batdongsan.com.vn,https://batdongsan.com.vn/ban-nha-dat-ha-noi,https://batdongsan.com.vn/ban-shophouse-nha-ph...,CĐT bán shop LK từ 90m2 Vin Cổ Loa trục cầu Tứ...,90 m²,"12,4 tỷ"


In [4]:
import pandas as pd

df_check = pd.read_csv("../Data/raw/crawled_hanoi.csv")
print(f"Tổng số tin duy nhất đã crawl: {len(df_check):,} tin")
print(df_check["Nguồn dữ liệu"].value_counts())

Tổng số tin duy nhất đã crawl: 1,152 tin
Nguồn dữ liệu
Crawl_Alonhadat.com.vn     1102
Crawl_Batdongsan.com.vn      50
Name: count, dtype: int64
